### Modeling Earth's Climate (Pfalzgraff, Eklof & Neshyba, 2026)
# Cambio 2.0

## Equations of motion of Cambio2.0
The equations of motion of Cambio2.0 are almost the same as Cambio1.0. The sole difference is that Cambio2.0 takes into account the *van't Hoff effect*, by which the temperature dependence of the solubility of carbon dioxide in the ocean is taken into account. This effect is implemented in Eq. 3 below:

$$
F_{la} =  k_{la} \ \ \ (1) 
$$

$$
F_{al} = k_{al0} +  k_{al1} \times C_{atm} \ \ \ (2)
$$

$$
F_{oa} = k_{oa} \times (1+DC\times T_{anomaly})  C_{ocean} \ \ \ (3)
$$

$$
F_{ao} = k_{ao} C_{atm} \ \ \ (4)
$$

$$
F_{ha} = \epsilon(t) \ \ \ (5)
$$

As previously, we'll solve this problem numerically by setting up a loop that adjusts the amount of carbon in the reservoirs at each time step, 

$$
C_{atm,n+1} = C_{atm,n}+(F_{la}+F_{oa}+F_{ha}-F_{al}-F_{ao}) \times \Delta t \ \ \ (6)
$$

$$
C_{ocean,n+1} = C_{ocean,n}+(F_{oa}-F_{ao}) \times \Delta t \ \ \ (7)
$$

This innovation is a first step toward incorporating positive feedbacks into Cambio (of which more to come later).

## Python development
We'll be developing a couple of Python-specific ideas here. One is the use of the Python "help" function, which can be very useful when developing new Python functions. A second idea is the *ClimateState* dictionary, which we'll use to package up all the variables that define the current state of the climate (it's cleaner this way). And a third idea is to develop two new functions that solve the Cambio equations of motion. These are called *PropagateClimateState_Cambio1* and *PropagateClimateState_Cambio2*), and they will help us expand the capabilities of our model.

## Uploading your climate emissions scenario
As before, you'll need to upload a climate emissions scenario file to the current folder.

## Learning goals
1. I can make effective use of Python's "help" command.
1. I can explain how the *van't Hoff effect* plays a role in these equations, and in their results.
1. I can make lists of Python dictionaries.
2. I can describe what is going on when the argument list of a Python function contains the name of another Python function.

In [ ]:
%pip install -q ipympl

In [ ]:
%matplotlib widget

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sys
sys.path.append('..')
import MECLib as cl
from copy import copy as makeacopy

### Python's "help" command
The cell below shows how to get details about functions. Try it!

In [ ]:
help(np.log10)
help(cl.Diagnose_OceanSurfacepH)

<!-- BEGIN QUESTION -->

### Your turn
Use Python's "help" command to get help on these functions in the cl library: 
- cl.Diagnose_OceanSurfacepH
- cl.Diagnose_actual_temperature
- cl.Diagnose_T_anomaly)

_Points:_ 3

In [ ]:
...

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### Loading the scheduled flow
In the cell below, load in the scheduled flows file you have uploaded. It'll be most convenient if you use the following naming convention: 

    time, eps, epsdictionary_fromfile = cl.LoadMyScenario(filename,verbose=True)

(but of course supplying the name of your own scheduled flows file).

_Points:_ 2

In [ ]:
...

<!-- END QUESTION -->

### Creating a dictionary for climate parameters
In the cell below, we use the CreateClimateParams function to create a dictionary of climate parameters.

In [ ]:
ClimateParams = cl.CreateClimateParams(epsdictionary_fromfile)
display(ClimateParams)

### Creating a climate state
Below we use the climate parameters we just created to create a pre-industrial climate state.

In [ ]:
ClimateState = cl.CreateClimateState(ClimateParams)
display(ClimateState)

### Making a list of climate states
Eventually, we're going to want to make list of climate states. In the next cell, we practice that process by making a very simple and boring list -- two identical preindustrial climate states. 

You might notice that the syntax for this is different from how we appended to make lists before. That's because here we're creating a regular Python list, whereas before we were creating a list of numpy arrays.

In [ ]:
ClimateState_list = []
print('An empty list:')
display(ClimateState_list)

print('')
print('A list with one climate state:')
ClimateState_list.append(ClimateState)
display(ClimateState_list)

print('')
print('A list with two climate states:')
ClimateState_list.append(ClimateState)
display(ClimateState_list)

### Extracting a time series from a climate list
The "display" capability we just used is handy to get a view of the contents of a list, but sometimes we're more interested in particular items in the list. The function CollectClimateTimeSeries does just that. In the cell below, you can see how it works. Of course there are just two values, but when we use CollectClimateTimeSeries later, there will be hundreds of values.

In [ ]:
time_array = cl.CollectClimateTimeSeries(ClimateState_list,'time')
display(time_array)

<!-- BEGIN QUESTION -->

### Your turn
Use cl.CollectClimateTimeSeries to collect and display (or print) all the atmospheric reservoir carbon amounts (C_atm) in ClimateState_list.

_Points:_ 2

In [ ]:
...

<!-- END QUESTION -->

### A new architecture for Cambio
The cell below duplicates Cambio1.0's algorithm for propagating the climate state forward. This is all done for you, but it's worth taking a few minutes to study it to understand how it works. So items worth paying particular attention to:

- There's a "propagator" function, *PropagateCS_Cambio1*, whose job is to move a starting climate state (*previousClimateState*) forward a short amount of time, as controlled by the climate parameters in *ClimateParams*,  the duration of time *dt*, and under the influence of anthropogenic flux *F_ha* at a specific time. The function returns the new climate state. It's assumed that *dt* is short enough that the simple flux expressions that you used previously inside your Euler loop, like "C_atm += (F_la + F_oa - F_ao - F_al + F_ha)*dt", are accurate enough for our purposes.
  
- There's another function, *run_Cambio*, whose job is to propagate the *ClimateState* forward as many times as there are time steps in your emission scenario. *run_Cambio* returns a *list* of climate states, one for each time step. One of the arguments of *run_Cambio* is the propagator function, *PropagateCS_Cambio1*. (Later, we'll re-use *run_Cambio*,  with other propagators as input arguments, like *PropagateCS_Cambio2*.)
- There's yet one more function, *CS_list_plots*, whose job it is to plot specific items in the list of climate state generated by *run_Cambio*. *CS_list_plots* also reports out maxima and minima of the arrays being plotted, which can be pretty handy.

In [ ]:
def PropagateCS_Cambio1(previousClimateState, ClimateParams, dt, F_ha):
    """Propagates the state of the climate, with a specified anthropogenic carbon flux"""
    """Returns a new climate state"""

    # Extract constants from ClimateParams
    k_la = ClimateParams['k_la']
    k_al0 = ClimateParams['k_al0']
    k_al1 = ClimateParams['k_al1']
    k_oa = ClimateParams['k_oa']
    k_ao = ClimateParams['k_ao']
    DC = ClimateParams['DC']
    preindustrial_albedo = ClimateParams['preindustrial albedo']
    fractional_albedo_floor = ClimateParams['fractional_albedo_floor']
    albedo_Tstar = ClimateParams['albedo_Tstar']
    albedo_delta_T = ClimateParams['albedo_delta_T']
    k_al1_Tstar = ClimateParams['k_al1_Tstar']
    k_al1_deltaT = ClimateParams['k_al1_deltaT']
    fractional_k_al1_floor = ClimateParams['fractional_k_al1_floor']
    
    # Extract concentrations, albedo, etc, from the previous climate state
    C_atm = previousClimateState['C_atm']
    C_ocean = previousClimateState['C_ocean']
    albedo = previousClimateState['albedo']
    time = previousClimateState['time']
    
    # Get the temperature implied by the carbon in the atmosphere, and ocean pH
    T_anomaly = cl.Diagnose_T_anomaly(C_atm, preindustrial_albedo, ClimateParams)
    actual_temperature = cl.Diagnose_actual_temperature(T_anomaly)
    OceanSurfacepH = cl.Diagnose_OceanSurfacepH(C_atm,ClimateParams)
    
    # Get new fluxes
    F_la = k_la    
    F_al = k_al0 + k_al1*C_atm
    F_oa = k_oa*C_ocean    
    F_ao = k_ao*C_atm

    # Get new concentrations of carbon that depend on the fluxes
    C_atm += (F_la + F_oa - F_ao - F_al + F_ha)*dt
    C_ocean += (F_ao - F_oa)*dt
    time += dt

    # Create a new climate state with these updates
    ClimateState = makeacopy(previousClimateState)
    ClimateState['C_atm'] = C_atm
    ClimateState['C_ocean'] = C_ocean
    ClimateState['F_al'] = F_al
    ClimateState['F_la'] = F_la
    ClimateState['F_ao'] = F_ao
    ClimateState['F_oa'] = F_oa
    ClimateState['F_ha'] = F_ha
    ClimateState['F_ocean_net'] = F_oa - F_ao
    ClimateState['F_land_net'] = F_la - F_al
    ClimateState['time'] = time
    ClimateState['T_anomaly'] = T_anomaly
    ClimateState['actual temperature'] = actual_temperature
    ClimateState['OceanSurfacepH'] = OceanSurfacepH
    ClimateState['albedo'] = albedo

    # Return the new climate state
    return ClimateState

# Run Cambio1.0
CS_Cambio1_list = cl.run_Cambio(PropagateCS_Cambio1, ClimateParams, time, eps)

# Choose items to plot
items_to_plot = [['C_atm','C_ocean'],['F_ha','F_ocean_net','F_land_net'],'T_anomaly','albedo']

# Plot those items
cl.CS_list_plots(CS_Cambio1_list,'Cambio1',items_to_plot)

<!-- BEGIN QUESTION -->

### Upgrading to Cambio2.0
Below, the idea is to tweak what we just did such that we include the van't Hoff effect. Here is a summary of the changes you'll need:
1. Don't forget to name your function *PropagateClimateState_Cambio2* (not *PropagateClimateState_Cambio1*)
1. Inside *PropagateClimateState_Cambio2*, use the temperature anomaly to implement the van't Hoff effect -- as represented by Eq. 3 in the Introduction.
1. *PropagateClimateState_Cambio2* also needs to be passed on to *run_Cambio*.
1. In addition to plotting the timelines you made for Cambio1.0, modify the variable *items_to_plot* to include 'OceanSurfacepH'.

_Points:_ 12

In [ ]:
...

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### Pause for analysis
1. Comparing Cambio 1.0 to 2.0 (i.e., the effect of incorporating the effect of temperature on the solubility of CO2 in the oceans), how much warmer is the peak temperature according to your Cambio 2.0 results vs 1.0? 
1. Focusing on just Cambio 2.0, what's the most acidic ocean pH?

_Points:_ 2

_Type your answer here, replacing this text._

<!-- END QUESTION -->

### Validating and finishing up
Assuming all this has gone smoothly, don't forget to do a Kernel/Restart & Run All, Validate, and submit.